# Tahap 2 - Preprocessing Teks Bahasa Indonesia

Notebook ini digunakan untuk dokumentasi tahap preprocessing pada project skripsi **Analisis Sentimen pada Aplikasi JakOne Mobile Menggunakan Metode IndoBERT**.

Tahap ini hanya membersihkan teks ulasan. Tidak ada labeling sentimen, split dataset, training model, evaluasi, atau visualisasi hasil.

## Input dan Output

Input yang digunakan berasal dari tahap pengumpulan data:

```text
data/raw/jakone_reviews_raw.csv
```

Output tahap preprocessing disimpan ke:

```text
data/processed/jakone_reviews_clean.csv
```

## Langkah Preprocessing

Langkah yang dilakukan pada teks ulasan:

1. Menghapus review kosong.
2. Menghapus duplikasi berdasarkan `review_id`, dan memakai `review` sebagai cadangan jika `review_id` kosong.
3. Case folding.
4. Menghapus URL.
5. Menghapus mention.
6. Menghapus simbol hashtag tetapi mempertahankan katanya.
7. Menghapus emoji.
8. Menghapus simbol dan karakter tidak penting.
9. Normalisasi kata slang atau singkatan Bahasa Indonesia.
10. Menghapus stopword Bahasa Indonesia dengan tetap mempertahankan kata penting untuk konteks aplikasi perbankan.
11. Merapikan spasi.

In [ ]:
from pathlib import Path
import importlib.util

import pandas as pd

## Memuat Fungsi dari Script

Notebook menggunakan fungsi yang sama dengan script `src/02_preprocess_data.py` agar hasil preprocessing konsisten ketika dijalankan lewat notebook maupun terminal.

In [ ]:
module_path = Path("src/02_preprocess_data.py")
if not module_path.exists():
    module_path = Path("../src/02_preprocess_data.py")

spec = importlib.util.spec_from_file_location("preprocess_module", module_path)
preprocess_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(preprocess_module)

## Contoh Fungsi `clean_text`

Fungsi `clean_text()` membersihkan satu teks ulasan. Contoh berikut memperlihatkan perubahan dari teks mentah menjadi teks bersih.

In [ ]:
sample_text = "Aplikasi Sering ERROR Saat LOGIN!!! OTPnya gak masuk, cek www.contoh.com #jakone"
clean_sample = preprocess_module.clean_text(sample_text)

print("Sebelum:", sample_text)
print("Sesudah:", clean_sample)

## Membaca Dataset Mentah

In [ ]:
input_path = Path(preprocess_module.INPUT_PATH)
if not input_path.exists():
    input_path = Path("../") / preprocess_module.INPUT_PATH

df_raw = pd.read_csv(input_path)
preprocess_module.validate_columns(df_raw)

print(f"Jumlah data awal: {len(df_raw)}")
df_raw.head()

## Menjalankan Preprocessing

In [ ]:
(
    df_clean,
    total_initial,
    total_after_empty_review,
    total_after_deduplication,
    total_after_preprocessing,
) = preprocess_module.preprocess_reviews(df_raw)

print(f"Jumlah data awal: {total_initial}")
print(f"Jumlah data setelah hapus review kosong: {total_after_empty_review}")
print(f"Jumlah data setelah hapus duplikasi: {total_after_deduplication}")
print(f"Jumlah data setelah preprocessing: {total_after_preprocessing}")

## Contoh Data Sebelum dan Sesudah Preprocessing

In [ ]:
df_clean[["review", "clean_review"]].head(10)

## Menyimpan Dataset Bersih

In [ ]:
output_path = Path(preprocess_module.OUTPUT_PATH)
if not output_path.is_absolute() and input_path.parts[0] == "..":
    output_path = Path("../") / output_path

output_path.parent.mkdir(parents=True, exist_ok=True)
df_clean.to_csv(output_path, index=False, encoding="utf-8-sig")
print(f"Data hasil preprocessing disimpan ke: {output_path}")